# Solvability classifier

Train on `train-3000` + `abstention-pack`. Report metrics on that mix and on `test-100`. `test-100` is not used to fit the model or to overwrite `tau.json`.


In [3]:
%pip install -q scikit-learn==1.6.1


Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import importlib.util
import json

SKLEARN_PIN = "scikit-learn==1.6.1"

# Kaggle mounts this dataset as
# /kaggle/input/datasets/<user>/<slug>/<pack>/<pack>/<task>/task.json
# Local runs still use task_packs/.
SLUGS = {
    "train": ("train-3000", "train_3000"),
    "abstention": ("abstention-pack", "abstention_pack"),
    "test": ("test-100", "test_100"),
}


def _repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "harness" / "features.py").is_file() and (cand / "task_packs").is_dir():
            return cand
    return None


def _task_jsons(root: Path) -> list[Path]:
    direct = sorted(p for p in root.glob("*/task.json") if p.is_file())
    if direct:
        return direct
    return sorted(p for p in root.glob("*/*/task.json") if p.is_file())


def _matches_slug(path: Path, slugs: tuple[str, ...]) -> bool:
    blob = path.as_posix().lower()
    return any(slug in blob for slug in slugs)


def find_pack(key: str) -> Path:
    slugs = SLUGS[key]
    roots = []
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        roots.append(kaggle)
    repo = _repo_root()
    if repo is not None:
        roots.append(repo / "task_packs")
    best = None
    best_n = -1
    for root in roots:
        if not root.is_dir():
            continue
        for manifest in root.rglob("manifest.jsonl"):
            parent = manifest.parent
            if not _matches_slug(parent, slugs):
                continue
            n = len(_task_jsons(parent))
            if n > best_n:
                best = parent
                best_n = n
    if best is None or best_n <= 0:
        raise FileNotFoundError(
            f"{key} pack not found under /kaggle/input or task_packs. "
            f"Expected a manifest.jsonl whose path contains one of {slugs}."
        )
    return best


"""Тут целый python file"""

_EMBEDDED_FEATURES = r'''"""Tabular features and structural skip flags. No LLM, no expected/reward.

Standalone: notebooks load this file by path, so it must not import the harness
package (that pulls openJiuwen).
"""

from __future__ import annotations

import ast
import re
from pathlib import Path
from typing import Any, Optional

FEATURE_SCHEMA_VERSION = 2   # bumped: new math_unsolvable_kw feature

# Bands match harness.estimate. The heuristic stays here so Kaggle and the
# runtime share one formula when no cost model is loaded.
COST_SHORT = 800
COST_CODING = 2500

FEATURE_NAMES: tuple[str, ...] = (
    "is_math",
    "is_data_analysis",
    "is_search",
    "is_coding",
    "instr_chars",
    "instr_words",
    "instr_lines",
    "digit_ratio",
    "n_numbers",
    "n_questions",
    "has_latex",
    "has_code_fence",
    "mentions_answer_txt",
    "da_named_file_missing",
    "da_no_csv",
    "da_broken_header",
    "da_column_missing",
    "search_kb_empty",
    "search_kb_tiny",
    "search_token_overlap",
    "coding_no_mylib",
    "coding_empty_test",
    "coding_foreign_import",
    "coding_ast_fail",
    "math_arith_steps",
    "math_unsolvable_kw",
    "csv_lines",
    "csv_cols",
    "kb_files",
    "kb_chars",
    "code_lines",
    "code_defs",
    "code_asserts",
)

# Hard skip only when a missing or wrong file makes the task impossible.
# A syntax error in mylib.py is a feature, not a skip: it may be the bug to fix.
# Tiny kb is a feature, not a skip: a short real note would be a false skip.
# math hard-skips only on explicit unsolvable-keyword rules; see _MATH_UNSOLVABLE_RE.
_HARD_SKIP_FLAGS: dict[str, tuple[str, ...]] = {
    "data_analysis": (
        "da_named_file_missing",
        "da_no_csv",
    ),
    "search": ("search_kb_empty",),
    "coding": (
        "coding_no_mylib",
        "coding_empty_test",
        "coding_foreign_import",
    ),
    "math": ("math_unsolvable_kw",),
}

_FILE_RE = re.compile(r"\b([\w./-]+\.(?:csv|tsv|json|txt|md|py))\b", re.IGNORECASE)
_COL_RE = re.compile(r"\bcolumn\s+([A-Za-z_][\w]*)", re.IGNORECASE)
_NUM_RE = re.compile(r"\d+(?:\.\d+)?")
_WORD_RE = re.compile(r"[A-Za-z]{3,}")
_IMPORT_RE = re.compile(
    r"^\s*(?:from|import)\s+([A-Za-z_][\w]*)",
    re.MULTILINE,
)
_ARITH_RE = re.compile(
    r"\b(?:split|double|doubled|add|added|receive|receives|ship|ships|"
    r"merge|merged|minus|plus|times|divided|subtract|multiply)\b",
    re.IGNORECASE,
)
_MATH_UNSOLVABLE_PATTERNS = (
    r"underdetermined",
    r"never\s+numerically\s+specified",
    r"exact\s+count\s+never\s+stated",
    r"unknown\s+number",
    r"undocumented\s+operations",
    r"contradictory\s+givens",
    r"must\s+be\s+both\s+.+?\s+and\s+.+?\s+at\s+once",
    r"write\s+NOTHING",
    r"if\s+the\s+problem\s+is\s+underdetermined",
)
_MATH_UNSOLVABLE_RE = re.compile("|".join(_MATH_UNSOLVABLE_PATTERNS), re.IGNORECASE)
_DEF_RE = re.compile(r"^\s*def\s+\w+", re.MULTILINE)
_ASSERT_RE = re.compile(r"\bassert\b|\bdef\s+test_\w+", re.IGNORECASE)

_STOP = {
    "the", "and", "for", "with", "that", "this", "from", "are", "was", "were",
    "you", "your", "only", "file", "answer", "when", "done", "write", "final",
    "current", "directory", "example", "bash", "echo", "python", "compute",
    "work", "inside", "using", "use", "what", "which", "who", "how", "many",
    "does", "have", "has", "into", "then", "than", "after", "before", "about",
}

_CSV_LINE_CAP = 5000
_KB_CHAR_CAP = 400_000
_KB_TINY_CHARS = 200


def _task_type(task: Any) -> str:
    raw = getattr(task, "task_type", "")
    return raw.value if hasattr(raw, "value") else str(raw)


def _work_dir(task: Any) -> Path:
    work = getattr(task, "work_dir", None)
    if work is not None:
        return Path(work)
    return Path(getattr(task, "task_dir"))


def _read_text(path: Path, cap: int) -> str:
    try:
        data = path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return ""
    return data[:cap]


def _named_files(instruction: str) -> list[str]:
    found: list[str] = []
    for match in _FILE_RE.findall(instruction):
        if Path(match).name.lower() == "answer.txt":
            continue
        found.append(match)
    return found


def _file_exists(work: Path, name: str) -> bool:
    if (work / name).is_file():
        return True
    return (work / Path(name).name).is_file()


def _table_paths(work: Path) -> list[Path]:
    paths: list[Path] = []
    for pattern in ("*.csv", "*.tsv", "*.json"):
        paths.extend(p for p in work.glob(pattern) if p.is_file())
    return sorted(paths)


def _csv_shape(path: Path) -> tuple[int, int, bool, list[str]]:
    """Return line count (capped), column count, broken-header flag, header cells."""
    try:
        fh = path.open("r", encoding="utf-8", errors="replace")
    except OSError:
        return 0, 0, True, []
    lines = 0
    header: list[str] = []
    with fh:
        for line in fh:
            if lines == 0:
                header = [cell.strip() for cell in line.rstrip("\n").split(",")]
            lines += 1
            if lines >= _CSV_LINE_CAP:
                break
    broken = lines == 0 or not any(header)
    cols = 0 if broken else len(header)
    return lines, cols, broken, header


def _kb_stats(work: Path) -> tuple[int, int, str]:
    kb = work / "kb"
    if not kb.is_dir():
        return 0, 0, ""
    files = sorted(p for p in kb.glob("*.md") if p.is_file())
    chunks: list[str] = []
    total = 0
    for path in files:
        text = _read_text(path, _KB_CHAR_CAP - total)
        if not text:
            continue
        chunks.append(text)
        total += len(text)
        if total >= _KB_CHAR_CAP:
            break
    body = "\n".join(chunks)
    nonempty = sum(1 for chunk in chunks if chunk.strip())
    return nonempty, len(body), body


def _token_overlap(instruction: str, kb_text: str) -> float:
    question = {w.lower() for w in _WORD_RE.findall(instruction) if w.lower() not in _STOP}
    if not question or not kb_text:
        return 0.0
    kb_words = {w.lower() for w in _WORD_RE.findall(kb_text)}
    return len(question & kb_words) / len(question)


def _coding_stats(work: Path) -> dict[str, float]:
    lib = work / "mylib.py"
    test = work / "test_mylib.py"
    lib_text = _read_text(lib, 200_000) if lib.is_file() else ""
    test_text = _read_text(test, 200_000) if test.is_file() else ""
    ast_fail = 0.0
    if lib.is_file():
        try:
            ast.parse(lib_text)
        except SyntaxError:
            ast_fail = 1.0
    imports = {name for name in _IMPORT_RE.findall(test_text)}
    foreign = 1.0 if imports and "mylib" not in imports else 0.0
    return {
        "coding_no_mylib": 0.0 if lib.is_file() else 1.0,
        "coding_empty_test": 0.0 if test_text.strip() else 1.0,
        "coding_foreign_import": foreign,
        "coding_ast_fail": ast_fail,
        "code_lines": float(lib_text.count("\n") + (1 if lib_text.strip() else 0)),
        "code_defs": float(len(_DEF_RE.findall(lib_text))),
        "code_asserts": float(len(_ASSERT_RE.findall(test_text))),
    }


def extract(task: Any) -> dict[str, float]:
    """Feature dict keyed by FEATURE_NAMES. Reward is intentionally absent."""
    task_type = _task_type(task)
    instruction = str(getattr(task, "instruction", "") or "")
    work = _work_dir(task)
    chars = len(instruction)
    words = re.findall(r"\S+", instruction)

    feats = {name: 0.0 for name in FEATURE_NAMES}
    feats["is_math"] = 1.0 if task_type == "math" else 0.0
    feats["is_data_analysis"] = 1.0 if task_type == "data_analysis" else 0.0
    feats["is_search"] = 1.0 if task_type == "search" else 0.0
    feats["is_coding"] = 1.0 if task_type == "coding" else 0.0
    feats["instr_chars"] = float(chars)
    feats["instr_words"] = float(len(words))
    feats["instr_lines"] = float(instruction.count("\n") + (1 if instruction else 0))
    feats["digit_ratio"] = (sum(ch.isdigit() for ch in instruction) / chars) if chars else 0.0
    feats["n_numbers"] = float(len(_NUM_RE.findall(instruction)))
    feats["n_questions"] = float(instruction.count("?"))
    feats["has_latex"] = 1.0 if ("$" in instruction or "\\frac" in instruction) else 0.0
    feats["has_code_fence"] = 1.0 if "```" in instruction else 0.0
    feats["mentions_answer_txt"] = 1.0 if "answer.txt" in instruction.lower() else 0.0
    feats["math_arith_steps"] = float(len(_ARITH_RE.findall(instruction)))
    feats["math_unsolvable_kw"] = 1.0 if (task_type == "math" and _MATH_UNSOLVABLE_RE.search(instruction)) else 0.0

    if task_type == "data_analysis":
        named = _named_files(instruction)
        if named and any(not _file_exists(work, name) for name in named):
            feats["da_named_file_missing"] = 1.0
        tables = _table_paths(work)
        if not tables:
            feats["da_no_csv"] = 1.0
        else:
            preferred = tables[0]
            for name in named:
                candidate = work / Path(name).name
                if candidate.is_file():
                    preferred = candidate
                    break
            if preferred.suffix.lower() in {".csv", ".tsv"}:
                lines, cols, broken, header = _csv_shape(preferred)
                feats["csv_lines"] = float(lines)
                feats["csv_cols"] = float(cols)
                feats["da_broken_header"] = 1.0 if broken else 0.0
                header_key = {cell.casefold() for cell in header if cell}
                missing_col = [
                    col for col in _COL_RE.findall(instruction)
                    if col.casefold() not in header_key
                ]
                if missing_col:
                    feats["da_column_missing"] = 1.0

    if task_type == "search":
        n_files, n_chars, body = _kb_stats(work)
        feats["kb_files"] = float(n_files)
        feats["kb_chars"] = float(n_chars)
        feats["search_kb_empty"] = 1.0 if n_files == 0 or not body.strip() else 0.0
        feats["search_kb_tiny"] = 1.0 if 0 < n_chars < _KB_TINY_CHARS else 0.0
        feats["search_token_overlap"] = _token_overlap(instruction, body)

    if task_type == "coding":
        feats.update(_coding_stats(work))

    return feats


def vector(task: Any) -> list[float]:
    feats = extract(task)
    return [float(feats[name]) for name in FEATURE_NAMES]


def structural_skip_reason(task: Any, feats: Optional[dict[str, float]] = None) -> Optional[str]:
    """Comma-joined hard-fail flags, or None. Math hard-fails only on explicit unsolvable-kw."""
    task_type = _task_type(task)
    flags = _HARD_SKIP_FLAGS.get(task_type)
    if not flags:
        return None
    row = feats if feats is not None else extract(task)
    hit = [name for name in flags if row.get(name, 0.0) >= 1.0]
    if not hit:
        return None
    return ",".join(hit)


def heuristic_cost(task: Any, feats: Optional[dict[str, float]] = None) -> int:
    """Same bands as harness.estimate.feature_adjust, from the feature row."""
    row = feats if feats is not None else extract(task)
    base = COST_CODING if row["is_coding"] >= 1.0 else COST_SHORT
    factor = 1.0
    if row["instr_chars"] > 800:
        factor *= 1.2
    if row["is_search"] >= 1.0 and row["kb_files"] > 100:
        factor *= 1.1
    if row["is_data_analysis"] >= 1.0 and row["csv_lines"] > 500:
        factor *= 1.1
    if row["is_coding"] >= 1.0 and row["coding_no_mylib"] < 1.0:
        factor *= 1.0 + min(row["code_lines"], 200.0) / 200.0
    return max(1, int(base * factor))
'''

def find_features() -> Path:
    candidates = []
    repo = _repo_root()
    if repo is not None:
        candidates.append(repo / "harness" / "features.py")
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        candidates.extend(kaggle.rglob("features.py"))
    for path in candidates:
        if not path.is_file():
            continue
        head = path.read_text(encoding="utf-8", errors="replace")[:800]
        if "FEATURE_SCHEMA_VERSION" in head:
            return path
    dest = Path("/kaggle/working/features.py")
    if not dest.parent.is_dir():
        dest = Path("features.py")
    dest.write_text(_EMBEDDED_FEATURES, encoding="utf-8")
    print("features.py is not in the Kaggle dataset; using the copy stored in this notebook")
    return dest


PACKS = {key: find_pack(key) for key in SLUGS}
FEATURES_PY = find_features()

spec = importlib.util.spec_from_file_location("rank_features", FEATURES_PY)
features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(features)
print("features", FEATURES_PY, "schema", features.FEATURE_SCHEMA_VERSION)
print({k: str(v) for k, v in PACKS.items()})


features.py is not in the Kaggle dataset; using the copy stored in this notebook
features /kaggle/working/features.py schema 2
{'train': '/kaggle/input/datasets/kulichenkoalexey/tasks-for-classifying/train-3000/train-3000', 'abstention': '/kaggle/input/datasets/kulichenkoalexey/tasks-for-classifying/abstention-pack/abstention-pack', 'test': '/kaggle/input/datasets/kulichenkoalexey/tasks-for-classifying/test-100/test-100'}


In [5]:
def load_pack(root: Path) -> list[dict]:
    manifest = {}
    with (root / "manifest.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            manifest[row["task_id"]] = row
    rows = []
    for task_json in _task_jsons(root):
        raw = json.loads(task_json.read_text(encoding="utf-8"))
        task_dir = task_json.parent
        work = task_dir / "workspace"

        class _Task:
            pass

        task = _Task()
        task.task_id = str(raw["task_id"])
        task.task_type = str(raw["task_type"])
        task.instruction = str(raw["instruction"])
        task.reward = int(raw["reward"])
        task.work_dir = work if work.is_dir() else task_dir
        feats = features.extract(task)
        meta = manifest.get(task.task_id, {})
        hard = features.structural_skip_reason(task, feats)
        rows.append({
            **feats,
            "task_id": task.task_id,
            "task_type": task.task_type,
            "reward": task.reward,
            "solvable": bool(meta.get("solvable", False)),
            "hard_skip": hard is not None,
            "hard_reason": hard or "",
            "cost_hat": features.heuristic_cost(task, feats),
        })
    return rows


train_rows = load_pack(PACKS["train"]) + load_pack(PACKS["abstention"])
test_rows = load_pack(PACKS["test"])
print(f"train-mix {len(train_rows)} solvable {sum(r['solvable'] for r in train_rows)}")
print(f"test-100 {len(test_rows)} solvable {sum(r['solvable'] for r in test_rows)}")


<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\W'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\*'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\.'
<unknown>:2: SyntaxWarning: invalid escape sequence '\.'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'


train-mix 3100 solvable 2400
test-100 100 solvable 80


In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

NAMES = list(features.FEATURE_NAMES)
train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)
X_train = train_df[NAMES]
y_train = train_df["solvable"].astype(int)

clf = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.08,
    max_iter=200,
    random_state=0,
)
clf.fit(X_train, y_train)
train_df["p"] = clf.predict_proba(X_train)[:, list(clf.classes_).index(1)]
test_df["p"] = clf.predict_proba(test_df[NAMES])[:, list(clf.classes_).index(1)]

baseline = LogisticRegression(max_iter=500)
baseline.fit(X_train, y_train)
train_df["p_base"] = baseline.predict_proba(X_train)[:, list(baseline.classes_).index(1)]
test_df["p_base"] = baseline.predict_proba(test_df[NAMES])[:, list(baseline.classes_).index(1)]
print("fitted", len(train_df), "rows")


fitted 3100 rows


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
def is_skip(row, p, tau):
    if bool(row.hard_skip):
        return True
    if row.task_type == "math":
        return False
    return float(p) < tau

def policy_skip(row, tau):
    if bool(row.hard_skip):
        return True
    if row.task_type == "math":
        return False
    return float(row.p) < tau


def skip_counts(df, p_col, tau):
    skipped = [is_skip(row, getattr(row, p_col), tau) for row in df.itertuples(index=False)]
    skipped = np.asarray(skipped)
    solvable = df["solvable"].to_numpy()
    false_skips = int((skipped & solvable).sum())
    caught = int((skipped & ~solvable).sum())
    return false_skips, caught, skipped


def choose_tau(df, p_col):
    scores = df[p_col].to_numpy()
    candidates = sorted(set([0.0, 1.0, *scores.tolist()]))
    best = None
    for tau in candidates:
        false_skips, caught, _ = skip_counts(df, p_col, tau)
        # Zero false skips first, then as many true skips as possible.
        key = (false_skips == 0, -false_skips, caught)
        if best is None or key > best[0]:
            best = (key, tau, false_skips, caught)
    return best[1], best[2], best[3]


def stricter_tau(df, p_col, tau):
    """Higher threshold that removes false skips on this frame. Not saved."""
    scores = sorted(set([tau, 1.0, *df[p_col].tolist()]))
    scores = [value for value in scores if value >= tau]
    found = None
    for candidate in scores:
        false_skips, caught, _ = skip_counts(df, p_col, candidate)
        if false_skips == 0 and (found is None or caught > found[1] or (caught == found[1] and candidate < found[0])):
            found = (candidate, caught)
    return found


tau, train_fp, train_caught = choose_tau(train_df, "p")
print(f"tau={tau:.6f} train false_skips={train_fp} caught_unsolvable={train_caught}")

import joblib
joblib.dump(
    {
        "model": clf,
        "feature_names": NAMES,
        "schema_version": features.FEATURE_SCHEMA_VERSION,
    },
    "solvability.joblib",
)
Path("tau.json").write_text(
    json.dumps({"tau": tau, "feature_schema_version": features.FEATURE_SCHEMA_VERSION}),
    encoding="utf-8",
)
print("wrote solvability.joblib and tau.json")


tau=0.514798 train false_skips=0 caught_unsolvable=643
wrote solvability.joblib and tau.json


## Metrics

False skip = a solvable task was not run. The goal is 0. Structural false skips are printed separately from the probability threshold.


In [9]:
def report(title, df, p_col, tau):
    false_skips, caught, skipped = skip_counts(df, p_col, tau)
    unsolvable = ~df["solvable"].to_numpy()
    predicted_unsolvable = skipped
    precision = caught / int(predicted_unsolvable.sum()) if predicted_unsolvable.any() else float("nan")
    recall = caught / int(unsolvable.sum()) if unsolvable.any() else float("nan")
    structural_fp = int(((df["hard_skip"]) & (df["solvable"]) & (df["task_type"] != "math")).sum())
    math_skips = int(sum(
        is_skip(row, getattr(row, p_col), tau)
        for row in df.itertuples(index=False)
        if row.task_type == "math"
    ))
    print(f"\n== {title} tau={tau:.6f} ==")
    print(f"false_skips={false_skips}  (goal 0)")
    print(f"unsolvable_precision={precision:.4f}  unsolvable_recall={recall:.4f}")
    print(f"structural_false_skips={structural_fp}  math_skips={math_skips}")
    frame = df.copy()
    frame["skip"] = skipped
    frame["pred_unsolvable"] = frame["skip"]
    print("by task_type:")
    for task_type, part in frame.groupby("task_type"):
        part_fp = int((part["skip"] & part["solvable"]).sum())
        part_caught = int((part["skip"] & ~part["solvable"]).sum())
        print(f"  {task_type}: false_skips={part_fp} caught={part_caught} n={len(part)}")
    matrix = pd.crosstab(
        frame["solvable"].map({True: "solvable", False: "unsolvable"}),
        frame["pred_unsolvable"].map({True: "skip", False: "run"}),
    )
    print(matrix)


report("train-mix HGB", train_df, "p", tau)
report("test-100 HGB", test_df, "p", tau)
test_fp, _, _ = skip_counts(test_df, "p", tau)
if test_fp:
    alt = stricter_tau(test_df, "p", tau)
    print("WARNING: test-100 has false skips. tau.json was NOT changed.")
    if alt is None:
        print("No stricter tau removes them. Structural skips are the cause if structural_false_skips > 0.")
    else:
        print(f"Stricter tau that zeroes test-100 false skips: {alt[0]:.6f} (caught {alt[1]}). Not saved.")

base_tau, base_fp, base_caught = choose_tau(train_df, "p_base")
print(f"\nlogistic baseline own tau={base_tau:.6f} train false_skips={base_fp} caught={base_caught}")
report("train-mix logistic", train_df, "p_base", base_tau)
report("test-100 logistic", test_df, "p_base", base_tau)



== train-mix HGB tau=0.514798 ==
false_skips=0  (goal 0)
unsolvable_precision=1.0000  unsolvable_recall=0.9186
structural_false_skips=0  math_skips=107
by task_type:
  coding: false_skips=0 caught=150 n=750
  data_analysis: false_skips=0 caught=143 n=750
  math: false_skips=0 caught=107 n=757
  search: false_skips=0 caught=243 n=843
pred_unsolvable   run  skip
solvable                   
solvable         2400     0
unsolvable         57   643

== test-100 HGB tau=0.514798 ==
false_skips=0  (goal 0)
unsolvable_precision=1.0000  unsolvable_recall=0.9500
structural_false_skips=0  math_skips=5
by task_type:
  coding: false_skips=0 caught=5 n=25
  data_analysis: false_skips=0 caught=4 n=25
  math: false_skips=0 caught=5 n=25
  search: false_skips=0 caught=5 n=25
pred_unsolvable  run  skip
solvable                  
solvable          80     0
unsolvable         1    19

logistic baseline own tau=0.043236 train false_skips=0 caught=507

== train-mix logistic tau=0.043236 ==
false_skips=0  (g

In [11]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def add_skip(df, p_col, tau):
    out = df.copy()
    out["skip"] = [
        is_skip(row, getattr(row, p_col), tau)
        for row in df.itertuples(index=False)
    ]
    return out

# 1. Сколько caught — из hard_skip, а сколько — из модели?
for title, df, p_col, t in [
    ("train", train_df, "p", tau),
    ("test",  test_df,  "p", tau),
]:
    frame = add_skip(df, p_col, t)
    hard_caught  = frame[frame["hard_skip"] & ~frame["solvable"]]
    model_caught = frame[frame["skip"] & ~frame["hard_skip"] & ~frame["solvable"]]
    hard_fp      = frame[frame["hard_skip"] & frame["solvable"]]
    print(f"{title}: hard_skip_caught={len(hard_caught)} "
          f"model_caught={len(model_caught)} hard_fp={len(hard_fp)}")

# 2. AUC / PR-AUC / Brier на test (для 'unsolvable' как positive)
y = (~test_df["solvable"]).astype(int)   # 1 = unsolvable
for name, col in [("HGB", "p"), ("logistic", "p_base")]:
    score = 1.0 - test_df[col].to_numpy()  # higher = more likely unsolvable
    print(name,
          "AUC=%.3f" % roc_auc_score(y, score),
          "AP=%.3f"  % average_precision_score(y, score),
          "Brier=%.3f" % brier_score_loss(y, score))

# 3. Per-type unsolvable в test
print(test_df.groupby(["task_type", "solvable"]).size().unstack(fill_value=0))

train: hard_skip_caught=357 model_caught=286 hard_fp=0
test: hard_skip_caught=13 model_caught=6 hard_fp=0
HGB AUC=0.996 AP=0.987 Brier=0.010
logistic AUC=0.976 AP=0.967 Brier=0.016
solvable       False  True 
task_type                  
coding             5     20
data_analysis      5     20
math               5     20
search             5     20


In [12]:
# 1. Hard-skip vs model в разрезе task_type — самый важный срез
for title, df, p_col, t in [("train", train_df, "p", tau), ("test", test_df, "p", tau)]:
    frame = add_skip(df, p_col, t)
    rows = []
    for tt, g in frame.groupby("task_type"):
        hard_c = ((g["hard_skip"]) & (~g["solvable"])).sum()
        model_c = ((g["skip"]) & (~g["hard_skip"]) & (~g["solvable"])).sum()
        total_u = (~g["solvable"]).sum()
        rows.append({"type": tt, "unsolvable": total_u,
                     "hard_caught": int(hard_c), "model_caught": int(model_c),
                     "model_recall_of_soft": round(model_c / max(total_u - hard_c, 1), 3)})
    print(f"\n{title}"); print(pd.DataFrame(rows).to_string(index=False))

# 2. Распределение p — понять, насколько "острые" вероятности
print(test_df["p"].describe())
print((test_df["p"] < 0.05).sum(), (test_df["p"] > 0.95).sum(), "of", len(test_df))

# 3. K каких задач в test ошиблась модель (p>tau, но unsolvable)
frame = add_skip(test_df, "p", tau)
missed = frame[~frame["skip"] & ~frame["solvable"]]
print(missed[["task_id", "task_type", "p", "hard_reason"]])
# посмотреть их task.json глазами — понять, есть ли сигнал, который фичи не ловят

# 4. OOF-AUC на train для честной оценки
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
oof = cross_val_predict(clf, X_train, y_train, method="predict_proba", cv=5)[:, 1]
print("train OOF AUC=%.4f" % roc_auc_score(y_train, oof))
print("train in-sample AUC=%.4f" % roc_auc_score(y_train, train_df["p"]))


train
         type  unsolvable  hard_caught  model_caught  model_recall_of_soft
       coding         150          150             0                  0.00
data_analysis         150           50            93                  0.93
         math         157          107             0                  0.00
       search         243           50           193                  1.00

test
         type  unsolvable  hard_caught  model_caught  model_recall_of_soft
       coding           5            5             0                 0.000
data_analysis           5            2             2                 0.667
         math           5            5             0                 0.000
       search           5            1             4                 1.000
count    100.000000
mean       0.807016
std        0.392623
min        0.000061
25%        0.994631
50%        0.999704
75%        0.999868
max        0.999994
Name: p, dtype: float64
19 77 of 100
                   task_id      task_typ

In [13]:
math_missed = add_skip(test_df, "p", tau)
math_missed = math_missed[(~math_missed["skip"]) & (~math_missed["solvable"]) & (math_missed["task_type"] == "math")]
print(math_missed[["task_id", "p"]])
# и посмотреть их instruction в task.json глазами

Empty DataFrame
Columns: [task_id, p]
Index: []
